# Day 32: Streaming & Async LLM Calls

This notebook demonstrates streaming with OpenAI, then builds a FastAPI server with a frontend.

In [ ]:
import os
from openai import AsyncOpenAI
from dotenv import load_dotenv
load_dotenv()

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Basic streaming with OpenAI

In [ ]:
async def stream_openai(prompt: str):
    stream = await client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    async for chunk in stream:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

# Test: collect chunks (in notebook we can't stream visually, but we can print)
import asyncio
async def test():
    full = ""
    async for token in stream_openai("Say hello"):
        full += token
        print(token, end="", flush=True)
    print("\nDone")
    return full
asyncio.run(test())

## 2. FastAPI app with StreamingResponse

Create a file `main.py` with the following code. Run with `uvicorn main:app --reload`.

In [ ]:
%%writefile main.py
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, HTMLResponse
from fastapi.templating import Jinja2Templates
from openai import AsyncOpenAI
import os
from dotenv import load_dotenv

load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
app = FastAPI()

async def generate_stream(prompt: str):
    stream = await client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    async for chunk in stream:
        if chunk.choices[0].delta.content:
            yield f"data: {chunk.choices[0].delta.content}\n\n"
    yield "data: [DONE]\n\n"

@app.get("/chat")
async def chat(request: Request):
    prompt = request.query_params.get("prompt", "")
    if not prompt:
        return {"error": "No prompt provided"}
    return StreamingResponse(generate_stream(prompt), media_type="text/event-stream")

@app.get("/")
async def index():
    html = """
    <!DOCTYPE html>
    <html>
    <head><title>Streaming Chat</title></head>
    <body>
        <h1>Streaming Chat</h1>
        <input type="text" id="prompt" size="50" placeholder="Ask something...">
        <button onclick="send()">Send</button>
        <div id="response" style="border:1px solid #ccc; margin-top:20px; padding:10px; min-height:100px;"></div>
        <script>
            async function send() {
                const prompt = document.getElementById('prompt').value;
                const responseDiv = document.getElementById('response');
                responseDiv.innerHTML = '';
                const eventSource = new EventSource(`/chat?prompt=${encodeURIComponent(prompt)}`);
                eventSource.onmessage = function(event) {
                    if (event.data === '[DONE]') {
                        eventSource.close();
                        return;
                    }
                    responseDiv.innerHTML += event.data;
                };
                eventSource.onerror = function() { eventSource.close(); };
            }
        </script>
    </body>
    </html>
    """
    return HTMLResponse(html)
print("File main.py created. Run: uvicorn main:app --reload")